# 01 · Трекинг I: как наблюдения становятся траекториями
Лекция 7. Восемь стендов, CPU. На каждом: сначала теория, затем исполняемый стенд и задания «прогноз → изменение → число → объяснение».

Все сцены и признаки стендов синтетические. Это учебные механизмы, **не полная реализация SORT/DeepSORT**. В SORT состояние 7D `[u,v,s,r,du,dv,ds]`, в DeepSORT 8D `[u,v,a,h,du,dv,da,dh]`. Первый стенд использует только 1D `[position,velocity]`; самостоятельная работа —2D `[x,y,vx,vy]` фиксированной рамки.

Пакеты: numpy, scipy, matplotlib, ipywidgets. Запустите в JupyterLab или Colab; GitHub не выполняет управление.

In [ ]:
# Запускайте первой. В Colab сохраняем штатные PyTorch/CUDA и ipywidgets.
import importlib.util, subprocess, sys
if sys.version_info < (3, 10):
    raise RuntimeError('Нужен Python 3.10 или новее.')
_packages = {'numpy': 'numpy>=1.26,<3', 'matplotlib': 'matplotlib>=3.8,<4', 'scipy': 'scipy>=1.11,<2', 'ipywidgets': 'ipywidgets>=7.7,<9'}
_missing = [spec for module, spec in _packages.items()
            if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])
    raise RuntimeError('Пакеты установлены. Перезапустите среду выполнения и выполните ноутбук сверху.')
import numpy, matplotlib
if not (1 <= int(numpy.__version__.split('.')[0]) < 3):
    raise RuntimeError('Нужен NumPy 1.x или 2.x; используйте отдельное окружение.')
if int(matplotlib.__version__.split('.')[0]) != 3:
    raise RuntimeError('Нужен Matplotlib 3.x; используйте отдельное окружение.')
_versions = {'python': sys.version.split()[0], 'numpy': numpy.__version__,
             'matplotlib': matplotlib.__version__}
if 'scipy' in _packages:
    import scipy
    if int(scipy.__version__.split('.')[0]) != 1:
        raise RuntimeError('Нужен SciPy 1.x; используйте отдельное окружение.')
    _versions['scipy'] = scipy.__version__
if 'cv2' in _packages:
    import cv2
    _versions['opencv'] = cv2.__version__
    _required_features = ()
    _absent = [name for name in _required_features if not hasattr(cv2, name)]
    if _absent:
        raise RuntimeError('В OpenCV нет ' + ', '.join(_absent) +
                           '. Установите один вариант OpenCV 4.12.0.88 в отдельной среде и перезапустите ядро.')
if 'ipywidgets' in _packages:
    import ipywidgets
    _versions['ipywidgets'] = ipywidgets.__version__
    if int(ipywidgets.__version__.split('.')[0]) not in (7, 8):
        raise RuntimeError('Нужен ipywidgets 7.x или 8.x; в Colab оставьте совместимую штатную версию.')
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
print('Окружение:', _versions)
print('Вычисления этой лабораторной: CPU; GPU не требуется.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Ellipse
from scipy.optimize import linear_sum_assignment
from pathlib import Path
import time,json,csv,itertools
plt.rcParams.update({'figure.dpi':110,'font.size':10})
np.set_printoptions(precision=4,suppress=True)

In [ ]:
import ipywidgets as W
from IPython.display import display
from IPython.utils.capture import capture_output
PANELS={}
def panel(name, fn, controls):
    out=W.Output(); state={'calls':0,'last':None}
    def refresh(change=None):
        with capture_output(display=True) as captured:
            result=fn(**{key:control.value for key,control in controls.items()})
        items=[{'output_type':'display_data','data':item.data,'metadata':item.metadata}
               for item in captured.outputs]
        if captured.stdout:
            items.append({'output_type':'stream','name':'stdout','text':captured.stdout})
        if captured.stderr:
            items.append({'output_type':'stream','name':'stderr','text':captured.stderr})
        out.outputs=tuple(items)
        state['last']=result; state['calls']+=1
    for control in controls.values():
        if hasattr(control,'continuous_update'): control.continuous_update=True
        control.observe(refresh,names='value')
    button=W.Button(description='Повторить опыт',icon='refresh')
    button.on_click(refresh)
    PANELS[name]={'controls':controls,'state':state,'refresh':refresh,'button':button}
    display(W.VBox([W.HBox(list(controls.values()),
                           layout=W.Layout(flex_flow='row wrap')),button,out]))
    refresh()

## 1. Predict, update и отсутствие детекции
$x^-=Fx$, $P^-=FPF^T+Q$. Для детекции $z$: innovation $z-Hx^-$, $S=HP^-H^T+R$, $K=P^-H^TS^{-1}$, $x=x^-+K(z-Hx^-)$. Q — неопределенность динамики, R — дисперсия измерения. При miss пропускается **update**, predict остается.

In [ ]:
def show_kalman(R,missing,z):
 x=np.array([10.,2.]);P=np.diag([4.,1.]);F=np.array([[1.,1.],[0.,1.]])
 xp=F@x;Pp=F@P@F.T+np.diag([1.,.25]);S=Pp[0,0]+R;K=Pp[:,0]/S
 xu=xp if missing else xp+K*(z-xp[0]);Pu=Pp if missing else Pp-np.outer(K,Pp[0])
 fig,ax=plt.subplots(figsize=(8,3))
 for ypos,mu,var,label,col in [(2,x[0],P[0,0],'previous','gray'),(1,xp[0],Pp[0,0],'predict','navy'),(0,xu[0],Pu[0,0],'after observation','seagreen')]:
  ax.errorbar(mu,ypos,xerr=2*np.sqrt(var),fmt='o',color=col,capsize=5,label=label)
 if not missing:ax.scatter([z],[0],marker='x',s=90,color='tomato',label='detection')
 ax.set(xlim=(1,24),yticks=[0,1,2],yticklabels=['after','predict','previous'],xlabel='position (pixels)',title='Bars: ±2 standard deviations');ax.legend(loc='upper right',ncol=1,fontsize=8);ax.grid(axis='x');plt.show();plt.close(fig)
 print(f'predict={xp}, Pxx={Pp[0,0]:.2f}; K={K}; after={xu}, Pxx={Pu[0,0]:.2f}; next predicted p={xu.sum():.2f}')
 return {'x':xu.tolist(),'Pxx':float(Pu[0,0]),'K':K.tolist()}
panel('kalman',show_kalman,{'R':W.FloatSlider(value=4,min=.25,max=20,step=.25,description='R variance',continuous_update=True),'missing':W.Checkbox(value=False,description='Нет детекции'),'z':W.FloatSlider(value=13,min=5,max=20,step=.5,description='z',continuous_update=True)})

### Проверка на стенде 1
1. **Прогноз:** для x=[10, 2], P=diag(4, 1), z 13, R 4 получите x−=[12, 2], K=[.6, .1], x=[12.6, 2.1]. Проверьте и вычислите следующее p 14.7.
2. Измените R 4→20 при том же z. Запишите K и Pxx, объясните, почему фильтр меньше доверяет детекции.
3. Включите «Нет детекции», затем меняйте z. Почему состояние не должно зависеть от несуществующего наблюдения? Полосы — ±2 стандартных отклонения, не рамки объекта.

## 2. Неопределённость и Mahalanobis gate
Для остатка $r=z-Hx^-$ оценка $d^2=r^T S^{-1}r$, где $S=HP^-H^T+R$. Здесь прогноз центра — $(0,0)$, $P^-=\operatorname{diag}(4,1)\cdot\text{P scale}$, $R=I$, наблюдение — $(z_x,1)$. Контур показывает $d^2=\text{threshold}$ в пространстве измерений. Gate запрещает ребро до глобального назначения; хорошая внешность не снимает запрет.

In [ ]:
def show_gate(zx,p_scale,threshold):
 mean=np.zeros(2);P=np.diag([4.,1.])*p_scale;R=np.eye(2)
 S=P+R;z=np.array([zx,1.]);residual=z-mean
 distance2=float(residual@np.linalg.solve(S,residual));allowed=distance2<=threshold
 theta=np.linspace(0,2*np.pi,240)
 rim=np.diag(np.sqrt(np.diag(S)*threshold))@np.vstack([np.cos(theta),np.sin(theta)])
 fig,ax=plt.subplots(figsize=(6,4));ax.plot(rim[0],rim[1],color='navy',label='Mahalanobis gate')
 ax.scatter([0],[0],color='navy',label='predicted centre');ax.scatter([zx],[1.],color='seagreen' if allowed else 'tomato',marker='x',s=100,label='detection')
 ax.plot([0,zx],[0,1.],ls=':',color='gray');ax.set(xlim=(-11,11),ylim=(-7,7),aspect='equal',xlabel='x',ylabel='y',title=f'd²={distance2:.3f}; threshold={threshold:.1f}; allowed={allowed}')
 ax.grid();ax.legend(loc='upper right');plt.show();plt.close(fig)
 print(f'S=diag({S[0,0]:.2f}, {S[1,1]:.2f}); d²={distance2:.3f}; allowed={allowed}. The same pixel displacement changes meaning when uncertainty changes.')
 return {'distance2':distance2,'allowed':bool(allowed),'S':S.tolist()}
panel('gate',show_gate,{'zx':W.FloatSlider(value=2,min=0,max=9,step=.25,description='z x',continuous_update=True),'p_scale':W.FloatSlider(value=1,min=.25,max=5,step=.25,description='P scale',continuous_update=True),'threshold':W.FloatSlider(value=4,min=.5,max=16,step=.5,description='gate d²',continuous_update=True)})

### Проверка на стенде 2
1. **Прогноз:** при $z_x=2$, P scale 1 и threshold 4 вычислите $S=\operatorname{diag}(5,2)$, $d^2=4/5+1/2=1.3$ и решение gate. Сверьте число.
2. Измените только $z_x$ с 2 на 5. Выпишите новое $d^2$ и решение. Почему тот же порог отвергает пару?
3. При $z_x=5$ измените только P scale с 1 на 4. Запишите $S$, $d^2$ и решение. Объясните, почему более неопределённый прогноз допускает большее смещение, хотя пиксельное расстояние не изменилось.

## 3. IoU — сходство рамок, не модель движения
$IoU=|A\cap B|/|A\cup B|$, cost=1−IoU. Координаты непрерывные xyxy, без +1 к ширине. Две далекие непересекающиеся рамки и две почти соприкасающиеся дают одинаковую IoU 0.

In [ ]:
def show_iou(dx,scale):
 a=np.array([0.,0.,4.,4.]);b=np.array([dx,1.,dx+4*scale,1+4*scale])
 inter=np.maximum(0,np.minimum(a[2:],b[2:])-np.maximum(a[:2],b[:2])).prod()
 union=16+(4*scale)**2-inter;iou=float(inter/union)
 fig,ax=plt.subplots(figsize=(6,3.5))
 for box,col,label in [(a,'navy','prediction'),(b,'tomato','detection')]:ax.add_patch(Rectangle(box[:2],*(box[2:]-box[:2]),fill=False,color=col,lw=2,label=label))
 ax.set(xlim=(-3,10),ylim=(-2,9),aspect='equal',title=f'IoU={iou:.3f}; cost 1−IoU={1-iou:.3f}');ax.legend();ax.grid();plt.show();plt.close(fig)
 print('IoU=0 не показывает расстояние между непересекающимися рамками; motion model может восстановить связь.')
 return {'iou':iou}
panel('iou',show_iou,{'dx':W.FloatSlider(value=1,min=-2,max=6,step=.5,description='dx',continuous_update=True),'scale':W.FloatSlider(value=1,min=.25,max=2,step=.25,description='scale',continuous_update=True)})

### Проверка на стенде 3
1. **Прогноз:** dx 1→3 при scale 1 уменьшит overlap. Измените dx, выпишите IoU и объясните.
2. При dx 0 измените scale 1→2: почему увеличенный bbox не получает IoU 1?
3. Поставьте dx 6. Объясните, какие дополнительные сведения нужны для связи при пропуске кадров.

## 4. Глобальное назначение, gate и unmatched
Назначение минимизирует общую стоимость, каждому track/detection — не более одной пары. Gate запрещает физически недопустимую связь **до** решения. Unmatched имеет цену: здесь u/2 за несопоставленный track и u/2 за несопоставленную detection. Поэтому одна связь конкурирует с отказом общей цены u.

In [ ]:
def tiny_assignment(cost,allowed,u):
 # Полный перебор только для стенда 2x2; основная работа должна решать N x M.
 best=(np.inf,[])
 for choices in itertools.product([-1,0,1],repeat=2):
  used=[j for j in choices if j>=0]
  if len(set(used))!=len(used) or any(j>=0 and not allowed[i,j] for i,j in enumerate(choices)):continue
  pairs=[(i,j) for i,j in enumerate(choices) if j>=0]
  value=sum(cost[i,j] for i,j in pairs)+u/2*(4-2*len(pairs))
  if value<best[0]:best=(value,pairs)
 return best

def show_assignment(case,gate,u):
 C=np.array([[1.,2.],[1.1,100.]]) if case=='greedy' else np.array([[.1,.2],[.11,.05]])
 allowed=np.ones((2,2),bool)
 if gate:allowed[1,1]=False
 score,pairs=tiny_assignment(C,allowed,u)
 ri,cj=linear_sum_assignment(C);post=[(int(i),int(j)) for i,j in zip(ri,cj) if allowed[i,j]]
 fig,ax=plt.subplots(figsize=(5,3));ax.imshow(C if case!='greedy' else np.log1p(C),cmap='Blues')
 for i,j in itertools.product(range(2),range(2)):
  ax.text(j,i,f'{C[i,j]:.2f}'+(' X' if not allowed[i,j] else ''),ha='center',va='center',color='red' if not allowed[i,j] else 'black')
 for i,j in pairs:ax.add_patch(Rectangle((j-.45,i-.45),.9,.9,fill=False,color='limegreen',lw=3))
 ax.set(xticks=[0,1],xticklabels=['D0','D1'],yticks=[0,1],yticklabels=['T0','T1'],title='Green: gated assignment with unmatched');plt.show();plt.close(fig)
 print(f'Correct pairs={pairs}; objective={score:.3f}; raw Hungarian then reject={post}')
 if case=='greedy':print('Greedy forced matching: 1+100=101; global full matching: 2+1.1=3.1.')
 return {'pairs':pairs,'posthoc':post,'objective':score}
panel('assignment',show_assignment,{'case':W.Dropdown(options=['greedy','gate trap'],value='gate trap',description='Сценарий'),'gate':W.Checkbox(value=True,description='Запрет T1-D1'),'u':W.FloatSlider(value=3,min=.1,max=6,step=.1,description='Unmatched',continuous_update=True)})

### Проверка на стенде 4
1. Сценарий greedy, gate выключен, u 6: прогнозируйте выбор; сравните 101 и 3.1. Почему минимальное локальное ребро не дает минимальной суммы?
2. Сценарий gate trap, gate включен, u 3: raw Hungarian выбирает запрещенную диагональ. После отбрасывания останется одна пара, хотя допустимы две перекрестные. Проверьте обе таблицы вывода.
3. Снизьте u до .1. Почему отказ от дорогих связей может быть лучше принудительного назначения?

У оригинального SORT есть собственная последовательность IoU/Hungarian/post-check. Этот общий учебный алгоритм явно моделирует gate и unmatched; не приписываем ему побитовую эквивалентность SORT.

## 5. Жизненный цикл: tentative → confirmed → deleted
Новый track получает hits 1. Для подтверждения требуется n_init последовательных наблюдений: в нашей схеме tentative уничтожается при первом miss. Confirmed сохраняется до misses>max_age. При удалении последующий объект создает новый ID.

In [ ]:
def show_lifecycle(n_init,max_age,early_miss):
 events=[True,False,True,True,False,False,False,True] if early_miss else [True,True,True,False,False,False,False,True]
 tracks=[];next_id=0;active=None;rows=[]
 for t,matched in enumerate(events):
  if active is None and matched:active={'id':next_id,'hits':1,'miss':0,'status':'confirmed' if n_init==1 else 'tentative'};next_id+=1
  elif active is not None:
   if matched:
    active['hits']+=1;active['miss']=0
    if active['hits']>=n_init:active['status']='confirmed'
   else:
    active['miss']+=1
    if active['status']=='tentative' or active['miss']>max_age:active['status']='deleted'
  state='none' if active is None else active['status'];tid=-1 if active is None else active['id'];rows.append((t,matched,tid,state))
  if state=='deleted':active=None
 fig,ax=plt.subplots(figsize=(9,2.7));colors={'none':'lightgray','tentative':'orange','confirmed':'seagreen','deleted':'tomato'}
 for t,matched,tid,state in rows:
  ax.scatter(t,0,c=colors[state],s=500,marker='s');ax.text(t,.22,f'ID {tid}\n{state}',ha='center',fontsize=8);ax.text(t,-.2,'hit' if matched else 'miss',ha='center')
 ax.set(xlim=(-.6,7.6),ylim=(-.5,.65),yticks=[],xlabel='frame');plt.show();plt.close(fig)
 print('Удаление confirmed происходит при misses > max_age; tentative удаляется после первого miss. Deleted ID не воскресает.')
 return {'rows':rows,'created':next_id}
panel('lifecycle',show_lifecycle,{'n_init':W.IntSlider(value=2,min=1,max=4,description='n_init',continuous_update=True),'max_age':W.IntSlider(value=2,min=0,max=5,description='max_age',continuous_update=True),'early_miss':W.Checkbox(value=False,description='Ранний пропуск')})

### Проверка на стенде 5
1. **Прогноз:** n_init2, max_age2 на исходной последовательности. На каком кадре track подтвержден и удален? Проверьте ряд.
2. Увеличьте max_age до 5. Что сохраняется через пропуск, и какой риск это создает для следующего объекта рядом?
3. Включите ранний пропуск. Сравните n_init1 и 3: почему одной ложной детекции не стоит сразу доверять? Состояние deleted не должно возвращаться в confirmed.

## 6. Внешность помогает и ошибается
Стоимость здесь смешивает нормированные motion и appearance. Движение в неоднозначной сцене предпочитает обмен ID, признаки могут поддержать правильную пару. В DeepSORT используются обученные descriptors, gallery, matching cascade и резервное IoU-сопоставление; простая сумма стенда не воспроизводит все это.

In [ ]:
def show_appearance(weight,bad_embedding,motion_gate):
 # Две пересекающиеся траектории: движение ошибочно предпочитает обмен ID.
 motion=np.array([[.3,.1],[.1,.3]])
 appearance=np.array([[.02,.8],[.8,.02]])
 if bad_embedding:appearance=appearance[:,::-1]
 C=(1-weight)*motion+weight*appearance;allowed=np.ones((2,2),bool)
 if motion_gate:allowed[0,0]=False
 score,pairs=tiny_assignment(C,allowed,1.)
 fig,axs=plt.subplots(1,3,figsize=(10,2.8))
 for ax,M,title in zip(axs,[motion,appearance,C],['motion cost','appearance cost','combined cost']):
  ax.imshow(M,vmin=0,vmax=1,cmap='Blues')
  for i,j in itertools.product(range(2),range(2)):ax.text(j,i,f'{M[i,j]:.2f}',ha='center',va='center')
  ax.set(xticks=[0,1],xticklabels=['A det','B det'],yticks=[0,1],yticklabels=['A track','B track'],title=title)
 for i,j in pairs:axs[2].add_patch(Rectangle((j-.45,i-.45),.9,.9,fill=False,color='limegreen',lw=3))
 if motion_gate:axs[2].text(0,0,'X',color='red',ha='center',fontsize=24)
 fig.tight_layout();plt.show();plt.close(fig)
 print(f'Pairs={pairs}; correct identity pairs={(0,0),(1,1)}. Appearance не отменяет motion gate.')
 return {'pairs':pairs}
panel('appearance',show_appearance,{'weight':W.FloatSlider(value=.5,min=0,max=1,step=.1,description='Appearance',continuous_update=True),'bad_embedding':W.Checkbox(value=False,description='Ошибочные признаки'),'motion_gate':W.Checkbox(value=False,description='Запрет A-A')})

### Проверка на стенде 6
1. **Прогноз:** weight 0→.5 при правильных признаках. Выпишите пары и момент смены решения.
2. Включите ошибочные признаки и увеличьте weight. Почему внешний признак не гарантирует сохранение identity?
3. Верните правильные признаки, weight 1, включите запрет A-A. Почему идеальный внешний score не должен обходить физический gate?

В основной работе сравнение выполняется на одинаковых наблюдениях и заранее заданных сценариях пересечения/пропуска.

## 7. Движение в SORT: прогноз, IoU, назначение и срок жизни ID

[SORT](https://arxiv.org/abs/1602.00763) хранит для каждой рамки состояние $[u,v,s,r,\dot u,\dot v,\dot s]$: центр, площадь, отношение сторон и три скорости. Перед каждым новым кадром фильтр Калмана **предсказывает** новую рамку. С ней сравнивают детекции по IoU; затем решают назначение и отвергают пары с недостаточным перекрытием. Непарная детекция создаёт новый ID. Трек без детекции может ещё оставаться внутри алгоритма, хотя его рамка уже не выдаётся наружу.

Здесь собственная учебная реализация этой последовательности на **синтетических** рамках. Она воспроизводит важные правила [авторского `sort.py`](https://github.com/abewley/sort/blob/master/sort.py), но численные матрицы фильтра, обработка вырожденных рамок и рисунки адаптированы для разбора. Цветные прямоугольники означают GT/наблюдения: **SORT не использует их цвет**. Метка GT известна только создателю сцены и оценщику; алгоритм видит координаты.

Запустите две ячейки ниже: сначала генератор сцены, затем стенд. Длинный код свернут для удобства; раскройте его, чтобы проследить этапы SORT и сравнить их с первоисточником.

Покадрово смотрите отдельно: серый контур истины, заливка детекции, пунктир прогноза, принятые пары и два списка ID — **внутренние** и **выданные**. На пересечении двух траекторий и при пропуске различие становится решающим.

In [ ]:
MOTION_PANELS = {}

def _scene_box(cx, cy, width=18., height=30.):
    return np.array([cx-width/2, cy-height/2, cx+width/2, cy+height/2], dtype=float)

def box_iou_matrix(A, B):
    """Rows A, columns B; continuous xyxy coordinates, with no pixel +1."""
    A = np.asarray(A, dtype=float).reshape(-1, 4)
    B = np.asarray(B, dtype=float).reshape(-1, 4)
    if len(A) == 0 or len(B) == 0:
        return np.zeros((len(A), len(B)), dtype=float)
    top_left = np.maximum(A[:, None, :2], B[None, :, :2])
    bottom_right = np.minimum(A[:, None, 2:], B[None, :, 2:])
    overlap = np.maximum(bottom_right-top_left, 0.)
    intersection = overlap[..., 0] * overlap[..., 1]
    area_a = np.prod(np.maximum(A[:, 2:]-A[:, :2], 0.), axis=1)
    area_b = np.prod(np.maximum(B[:, 2:]-B[:, :2], 0.), axis=1)
    union = area_a[:, None] + area_b[None, :] - intersection
    return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)

def rgb_feature(patch):
    """Measured mean RGB, L2 normalized; a toy appearance feature, not re-ID."""
    mean_rgb = np.asarray(patch, dtype=float).mean(axis=(0, 1))
    norm = np.linalg.norm(mean_rgb)
    return mean_rgb / norm if norm > 1e-12 else np.zeros(3, dtype=float)

def make_crossing_scene(miss_length=1, jitter=0.0, color_mode='distinct', false_positive=False):
    """Twelve fixed frames: only observations vary, while truth stays identical.

    GT 0 moves right and GT 1 left.  At frame 6 their x centres coincide.
    Detections are deliberately shuffled; list position never encodes identity.
    Patches are synthetic 4x4 RGB crops, so color is measurable but trivial.
    """
    if color_mode not in ('distinct', 'same', 'swapped', 'gray'):
        raise ValueError('color_mode: distinct, same, swapped or gray')
    miss_length = int(miss_length)
    if not 0 <= miss_length <= 4:
        raise ValueError('miss_length must be between 0 and 4')
    if float(jitter) < 0:
        raise ValueError('jitter must be nonnegative')
    missed = {0: set(), 1: {6}, 2: {5, 6}, 3: {5, 6, 7}, 4: {4, 5, 6, 7}}[miss_length]
    colors = (np.array([.85, .12, .08]), np.array([.08, .25, .85]))
    pattern = np.linspace(-.012, .012, 16).reshape(4, 4, 1)
    rng = np.random.default_rng(20261020)
    frames = []
    for t in range(12):
        truth = {0: _scene_box(14+6*t, 49), 1: _scene_box(86-6*t, 51)}
        detections = []
        for gt in (0, 1):
            if gt == 0 and t in missed:
                continue
            dx, dy = rng.normal(0, float(jitter), 2)
            box = truth[gt] + np.array([dx, dy, dx, dy])
            color_index = 1-gt if color_mode == 'swapped' and t >= 6 else gt
            base = colors[color_index] if color_mode in ('distinct', 'swapped') else (
                np.array([.52, .43, .24]) if color_mode == 'same' else np.array([.48, .48, .48]))
            patch = np.clip(np.broadcast_to(base, (4, 4, 3)) + pattern, 0, 1).copy()
            detections.append({'box': box, 'patch': patch, 'gt': gt})
        if false_positive and t in (3, 4):
            box = _scene_box(33 + 2*(t-3), 73)
            base = np.array([.55, .65, .18])
            patch = np.clip(np.broadcast_to(base, (4, 4, 3)) + pattern, 0, 1).copy()
            detections.append({'box': box, 'patch': patch, 'gt': -1})
        if t % 2:
            detections.reverse()
        frames.append({'truth': truth, 'detections': detections, 'frame': t})
    return frames

In [ ]:
def _sort_bbox_to_measurement(box):
    x1, y1, x2, y2 = np.asarray(box, dtype=float)
    width = max(x2-x1, 1e-6)
    height = max(y2-y1, 1e-6)
    return np.array([(x1+x2)/2, (y1+y2)/2, width*height, width/height])

def _sort_measurement_to_bbox(state):
    u, v, s, r = np.asarray(state, dtype=float)[:4]
    s, r = max(float(s), 1e-6), max(float(r), 1e-6)
    width = np.sqrt(s*r)
    height = s/width
    return np.array([u-width/2, v-height/2, u+width/2, v+height/2])

class _SortMotionTrack:
    def __init__(self, box, track_id):
        self.id = int(track_id)
        self.x = np.zeros(7, dtype=float)
        self.x[:4] = _sort_bbox_to_measurement(box)
        # Unobserved initial velocities have much larger uncertainty.
        self.P = np.diag([10., 10., 10., 10., 10000., 10000., 10000.])
        self.F = np.eye(7)
        self.F[0, 4] = self.F[1, 5] = self.F[2, 6] = 1.
        self.H = np.eye(7)[:4]
        self.Q = np.diag([1., 1., 1., .001, .01, .01, .0001])
        self.R = np.diag([1., 1., 10., 10.])
        self.time_since_update = 0
        self.hit_streak = 0
        self.hits = 0
        self.age = 0

    def bbox(self):
        return _sort_measurement_to_bbox(self.x)

    def predict(self):
        if self.x[2] + self.x[6] <= 1e-6:
            self.x[6] = 0.
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        self.age += 1
        if self.time_since_update > 0:
            self.hit_streak = 0
        self.time_since_update += 1
        return self.bbox()

    def update(self, box):
        measurement = _sort_bbox_to_measurement(box)
        innovation = measurement - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        gain = np.linalg.solve(S, (self.P @ self.H.T).T).T
        self.x = self.x + gain @ innovation
        residual = np.eye(7) - gain @ self.H
        self.P = residual @ self.P @ residual.T + gain @ self.R @ gain.T
        self.P = (self.P + self.P.T)/2
        self.time_since_update = 0
        self.hit_streak += 1
        self.hits += 1

class SortMotion:
    """Small independent teaching model of SORT's online decision sequence."""
    def __init__(self, iou_threshold=.3, max_age=1, min_hits=3, use_prediction=True):
        self.iou_threshold = float(iou_threshold)
        self.max_age = int(max_age)
        self.min_hits = int(min_hits)
        self.use_prediction = bool(use_prediction)
        self.tracks = []
        self.frame_count = 0
        self.next_id = 1

    def step(self, detections):
        self.frame_count += 1
        boxes = np.asarray([d['box'] for d in detections], dtype=float).reshape(-1, 4)
        previous = [track.bbox().copy() for track in self.tracks]
        predicted = []
        for track in self.tracks:
            predicted.append(track.predict().copy())
        match_boxes = predicted if self.use_prediction else previous
        matrix = box_iou_matrix(boxes, np.asarray(match_boxes).reshape(-1, 4))

        # The author code has a unique-above-threshold shortcut.  Otherwise it
        # assigns on the complete IoU matrix and rejects low-overlap pairs AFTER.
        if matrix.size:
            allowed = matrix > self.iou_threshold
            if allowed.sum(axis=1).max() == 1 and allowed.sum(axis=0).max() == 1:
                row, col = np.where(allowed)
            else:
                row, col = linear_sum_assignment(-matrix)
            candidate_pairs = [(int(i), int(j)) for i, j in zip(row, col)]
        else:
            candidate_pairs = []
        accepted = [(i, j) for i, j in candidate_pairs if matrix[i, j] >= self.iou_threshold]
        used_dets = {i for i, _ in accepted}
        used_tracks = {j for _, j in accepted}
        unmatched_dets = [i for i in range(len(boxes)) if i not in used_dets]
        unmatched_tracks = [self.tracks[j].id for j in range(len(self.tracks)) if j not in used_tracks]
        predicted_info = [
            {'id': int(track.id), 'box': predicted[j].copy(), 'match_box': np.asarray(match_boxes[j]).copy(),
             'time_since_update': int(track.time_since_update)}
            for j, track in enumerate(self.tracks)
        ]
        track_ids = [track.id for track in self.tracks]
        matches = []
        det_to_id = {}
        for i, j in accepted:
            track = self.tracks[j]
            track.update(boxes[i])
            det_to_id[int(i)] = int(track.id)
            matches.append((int(track.id), int(i), float(matrix[i, j])))
        created = []
        for i in unmatched_dets:
            track = _SortMotionTrack(boxes[i], self.next_id)
            self.next_id += 1
            self.tracks.append(track)
            created.append(track.id)
            det_to_id[int(i)] = int(track.id)
        emitted = []
        survivors = []
        deleted = []
        for track in self.tracks:
            if track.time_since_update < 1 and (track.hit_streak >= self.min_hits or self.frame_count <= self.min_hits):
                emitted.append({'id': int(track.id), 'box': track.bbox().copy()})
            if track.time_since_update > self.max_age:
                deleted.append(int(track.id))
            else:
                survivors.append(track)
        self.tracks = survivors
        internal = [
            {'id': int(t.id), 'box': t.bbox().copy(), 'age': int(t.age),
             'time_since_update': int(t.time_since_update), 'hit_streak': int(t.hit_streak)}
            for t in self.tracks
        ]
        return {'frame': self.frame_count-1, 'predicted': predicted_info, 'iou': matrix,
                'track_columns': track_ids, 'candidate_pairs': [(track_ids[j], i, float(matrix[i, j])) for i, j in candidate_pairs],
                'matches': matches, 'unmatched_dets': unmatched_dets,
                'unmatched_tracks': unmatched_tracks, 'created': created, 'deleted': deleted,
                'internal': internal, 'emitted': emitted, 'det_to_id': det_to_id}

def run_sort_scene(frames, iou_threshold=.3, max_age=1, min_hits=3, use_prediction=True):
    tracker = SortMotion(iou_threshold=iou_threshold, max_age=max_age,
                         min_hits=min_hits, use_prediction=use_prediction)
    decisions = []
    last_visible = {0: None, 1: None}
    id_changes = 0
    for frame in frames:
        decision = tracker.step(frame['detections'])
        visible = {item['id'] for item in decision['emitted']}
        gt_to_visible = {}
        for det_index, detection in enumerate(frame['detections']):
            gt = detection['gt']
            track_id = decision['det_to_id'].get(det_index)
            if gt in (0, 1) and track_id in visible:
                gt_to_visible[gt] = track_id
        for gt, track_id in gt_to_visible.items():
            if last_visible[gt] is not None and last_visible[gt] != track_id:
                id_changes += 1
            last_visible[gt] = track_id
        decision['gt_to_visible'] = gt_to_visible
        decision['id_changes'] = id_changes
        decisions.append(decision)
    return decisions

def _draw_sort_motion(frames, decisions, frame_index, show_truth=True, show_prediction=True):
    frame = frames[frame_index]
    decision = decisions[frame_index]
    fig, (scene_ax, matrix_ax) = plt.subplots(1, 2, figsize=(11.4, 4.4),
                                            gridspec_kw={'width_ratios': [1.8, 1]})
    colors = ['#166caa', '#d54b27', '#7c49a5', '#24945b', '#b47815', '#555555']
    def track_color(track_id):
        return colors[(int(track_id)-1) % len(colors)]
    if show_truth:
        for gt, box in frame['truth'].items():
            scene_ax.add_patch(Rectangle(box[:2], box[2]-box[0], box[3]-box[1],
                                         fill=False, edgecolor='0.6', linestyle=':', linewidth=1.5))
            scene_ax.text(box[0], box[3]+1, f'GT{gt}', fontsize=8, color='0.42')
    # Only emitted track centres form visible trails; internal predictions stay separate.
    for track_id in sorted({item['id'] for d in decisions[:frame_index+1] for item in d['emitted']}):
        points = [((item['box'][0]+item['box'][2])/2, (item['box'][1]+item['box'][3])/2)
                  for d in decisions[:frame_index+1] for item in d['emitted'] if item['id'] == track_id]
        if len(points) > 1:
            scene_ax.plot(*np.asarray(points).T, color=track_color(track_id), alpha=.55, linewidth=1.5)
    if show_prediction:
        for item in decision['predicted']:
            box = item['match_box']
            scene_ax.add_patch(Rectangle(box[:2], box[2]-box[0], box[3]-box[1],
                                         fill=False, edgecolor=track_color(item['id']),
                                         linestyle='--', linewidth=1.8))
            scene_ax.text(box[0], box[1]-2, f"P{item['id']}", fontsize=8,
                          color=track_color(item['id']))
    for i, detection in enumerate(frame['detections']):
        box = detection['box']
        rgb = np.asarray(detection['patch']).mean(axis=(0, 1))
        scene_ax.add_patch(Rectangle(box[:2], box[2]-box[0], box[3]-box[1],
                                     facecolor=rgb, alpha=.20, edgecolor=rgb,
                                     linewidth=2.3))
        scene_ax.text(box[0], box[1]+3, f'D{i}', fontsize=9, weight='bold', color='black')
    for item in decision['emitted']:
        box = item['box']
        cx, cy = (box[0]+box[2])/2, (box[1]+box[3])/2
        scene_ax.scatter([cx], [cy], s=60, color=track_color(item['id']),
                         edgecolor='white', zorder=4)
        scene_ax.text(cx+1, cy-4, f"ID {item['id']}", color=track_color(item['id']),
                      fontsize=9, weight='bold')
    scene_ax.set(xlim=(0, 100), ylim=(20, 92), xlabel='x, pixels', ylabel='y, pixels',
                 title=f'Кадр {frame_index}: детекции, прогноз, видимые ID')
    scene_ax.grid(alpha=.18)
    matrix = decision['iou']
    if matrix.size:
        matrix_ax.imshow(matrix, vmin=0, vmax=1, cmap='Blues')
        matrix_ax.set(xticks=np.arange(matrix.shape[1]),
                      xticklabels=[f"T{tid}" for tid in decision['track_columns']],
                      yticks=np.arange(matrix.shape[0]),
                      yticklabels=[f'D{i}' for i in range(matrix.shape[0])],
                      xlabel='предсказанный track', ylabel='детекция')
        matched_cells = {(det_i, decision['track_columns'].index(track_id))
                         for track_id, det_i, _ in decision['matches']}
        candidate_cells = {(det_i, decision['track_columns'].index(track_id))
                           for track_id, det_i, _ in decision['candidate_pairs']}
        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                mark = ' ✓' if (i, j) in matched_cells else (' ×' if (i, j) in candidate_cells else '')
                matrix_ax.text(j, i, f'{matrix[i, j]:.2f}{mark}', ha='center', va='center',
                               fontsize=9, color='white' if matrix[i, j] > .55 else 'black')
        matrix_ax.set_xlim(-.5, matrix.shape[1]-.5)
        matrix_ax.set_ylim(matrix.shape[0]-.5, -.5)
    else:
        matrix_ax.text(.5, .5, 'Нет пары detection × track', transform=matrix_ax.transAxes,
                       ha='center', va='center')
        matrix_ax.set(xticks=[], yticks=[])
    matrix_ax.set_title('IoU до обновления')
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    return decision

_sort_controls = {
    'frame': W.IntSlider(value=6, min=0, max=11, step=1, description='Кадр', continuous_update=True),
    'miss_length': W.IntSlider(value=1, min=0, max=4, step=1, description='Пропуск', continuous_update=True),
    'jitter': W.FloatSlider(value=0., min=0., max=3., step=.25, description='Шум xy', continuous_update=True),
    'iou_threshold': W.FloatSlider(value=.3, min=0., max=.8, step=.05, description='IoU ≥', continuous_update=True),
    'max_age': W.IntSlider(value=1, min=0, max=4, step=1, description='max_age', continuous_update=True),
    'min_hits': W.IntSlider(value=3, min=1, max=4, step=1, description='min_hits', continuous_update=True),
    'false_positive': W.Checkbox(value=False, description='Ложные детекции'),
    'use_prediction': W.Checkbox(value=True, description='Сопоставлять с прогнозом'),
    'show_truth': W.Checkbox(value=True, description='Показать GT'),
    'show_prediction': W.Checkbox(value=True, description='Показать прогноз')
}
_sort_output = W.Output()
_sort_state = {'calls': 0, 'last': None}
def _refresh_sort_motion(change=None):
    params = {key: control.value for key, control in _sort_controls.items()}
    frames = make_crossing_scene(params['miss_length'], params['jitter'],
                                 false_positive=params['false_positive'])
    decisions = run_sort_scene(frames, params['iou_threshold'], params['max_age'],
                               params['min_hits'], params['use_prediction'])
    index = params['frame']
    decision = decisions[index]
    with capture_output(display=True) as captured:
        _draw_sort_motion(frames, decisions, index, params['show_truth'], params['show_prediction'])
        print('Принятые пары (ID, D, IoU):', [(tid, det, round(score, 3)) for tid, det, score in decision['matches']])
        print('Кандидаты до порога:', [(tid, det, round(score, 3)) for tid, det, score in decision['candidate_pairs']])
        print('Непарные D:', decision['unmatched_dets'], 'непарные track:', decision['unmatched_tracks'])
        print('Созданы:', decision['created'], 'удалены:', decision['deleted'])
        print('Внутри:', [(item['id'], item['time_since_update'], item['hit_streak']) for item in decision['internal']],
              '(ID, кадров без update, серия hits)')
        print('Выданы:', [item['id'] for item in decision['emitted']],
              '| GT→выданный ID:', decision['gt_to_visible'],
              '| смены ID с начала сцены:', decision['id_changes'])
        print('До обновления IoU (строки D, столбцы track):',
              np.round(decision['iou'], 3).tolist())
        if not params['use_prediction']:
            print('Контрфакт: для сопоставления взята прошлая рамка; KF всё ещё прогнозирует состояние. Это не SORT.')
    items = [{'output_type': 'display_data', 'data': item.data, 'metadata': item.metadata}
             for item in captured.outputs]
    if captured.stdout:
        items.append({'output_type': 'stream', 'name': 'stdout', 'text': captured.stdout})
    if captured.stderr:
        items.append({'output_type': 'stream', 'name': 'stderr', 'text': captured.stderr})
    _sort_output.outputs = tuple(items)
    _sort_state['calls'] += 1
    _sort_state['last'] = {
        'frame': index, 'pairs': decision['matches'],
        'output_ids': [item['id'] for item in decision['emitted']],
        'internal_ids': [item['id'] for item in decision['internal']],
        'id_changes': decision['id_changes'], 'created': decision['created'],
        'deleted': decision['deleted'], 'iou': decision['iou'].tolist(),
        'gt_to_visible': decision['gt_to_visible']
    }

for _control in _sort_controls.values():
    _control.observe(_refresh_sort_motion, names='value')
_sort_play = W.Play(value=6, min=0, max=11, step=1, interval=650, description='▶')
_sort_play_link = W.link((_sort_play, 'value'), (_sort_controls['frame'], 'value'))
_sort_prev = W.Button(description='◀ Кадр')
_sort_next = W.Button(description='Кадр ▶')
_sort_reset = W.Button(description='С начала', icon='refresh')
def _sort_move(delta):
    before = _sort_controls['frame'].value
    _sort_controls['frame'].value = int(np.clip(before + delta, 0, 11))
    if _sort_controls['frame'].value == before:
        _refresh_sort_motion()
_sort_prev.on_click(lambda button: _sort_move(-1))
_sort_next.on_click(lambda button: _sort_move(1))
_sort_reset.on_click(lambda button: _sort_move(-_sort_controls['frame'].value))
MOTION_PANELS['sort'] = {
    'controls': _sort_controls, 'state': _sort_state,
    'buttons': {'prev': _sort_prev, 'next': _sort_next, 'reset': _sort_reset, 'play': _sort_play},
    'refresh': _refresh_sort_motion, 'output': _sort_output
}
display(W.VBox([
    W.HBox([_sort_play, _sort_prev, _sort_next, _sort_reset, _sort_controls['frame']]),
    W.HBox([_sort_controls[k] for k in ('miss_length', 'jitter', 'iou_threshold', 'max_age', 'min_hits')],
           layout=W.Layout(flex_flow='row wrap')),
    W.HBox([_sort_controls[k] for k in ('false_positive', 'use_prediction', 'show_truth', 'show_prediction')],
           layout=W.Layout(flex_flow='row wrap')),
    _sort_output
]))
_refresh_sort_motion()

### Проверка на стенде 7

1. **Прогноз → действие → число → смысл.** При одном пропуске и настройках по умолчанию дойдите до кадра 6. Запишите, какие ID существуют внутри и какие выданы наружу. Перейдите на кадр 7: какое назначение сделано после прогноза? Выключите «сопоставлять с прогнозом» и сравните матрицу IoU при тех же детекциях. Затем при пороге 0.50 сравните число созданных ID в обоих режимах. Режим без прогноза — контрфакт, а не SORT.
2. Зафиксируйте кадр 1 и сравните пороги IoU 0.3 и 0.55 при остальных исходных параметрах. Выпишите IoU правильной пары (около 0.50), принятые пары и новые ID. Почему порог после глобального назначения может оставить непарные объекты?
3. Установите `min_hits=1`, пропуск длиной 2 и сравните `max_age=1` и `max_age=3` на кадрах 6–8. Выпишите **внутренние/выданные** ID на кадре 8 и число смен ID. Почему сохранённый в памяти трек не равен видимой траектории? Затем верните `min_hits=3`: почему видимая смена ID появляется позже?
4. Включите две близкие ложные детекции и сравните `min_hits=1` с `min_hits=3`. Какие новые ID появились внутри, какие вышли наружу? Проверьте особое правило первых трёх кадров в SORT; не считайте любую временную рамку подтверждённой личностью.

Сохраните значения параметров, матрицы IoU и наблюдения: порядок детекций меняется между кадрами и не может служить ID. После этого стенда сравните **те же кадры** с цветовым признаком следующего стенда.

## 8. Цвет вместо обученного re-ID: движение, галерея и каскад

Одна и та же сцена движущихся рамок передаётся учебному SORT и второму трекеру. Второй повторяет **порядок решений Deep SORT**, но вместо обученной нейросети использует нормированный средний RGB маленькой цветовой вставки. Это намеренно слабый суррогат внешности: одинаковый цвет разных объектов и обесцвечивание могут его обмануть. Истинные метки A/B видны только на рисунке для разбора ошибок; трекер получает рамку и RGB, но не GT.

У цветового трекера состояние Калмана $(u,v,a,h,\dot u,\dot v,\dot a,\dot h)$, где $a=w/h$. После **predict** строится матрица $d^2_{ij}=(z_j-Hx_i^-)^TS_i^{-1}(z_j-Hx_i^-)$; порог $9.4877$ — 95% для четырёх координат рамки, а начальное значение ползунка $9.5$ — округление. Для подтверждённых треков стоимость $c_{ij}=\min_{r\in G_i}(1-r^T f_j)$, где $G_i$ — галерея последних цветовых признаков, $f_j$ — признак детекции. Недопустимая по движению пара исключается, даже если цвета совпали. Каскад сначала сопоставляет недавно увиденные треки; затем tentative и недавно не сопоставленные confirmed получают резервное сопоставление по IoU. После пропуска tentative удаляется сразу, confirmed — только когда возраст пропуска превысит `max_age`.

Переключатели «без каскада», «без gate» и «без резервного IoU» — **контрфактические учебные отключения**. Наша реализация использует простую калибровку ковариации и назначение с явной ценой отказа; это не запуск исходного кода Deep SORT. Длинный код следующей ячейки свернут: раскройте его и сравните порядок стадий с авторским трекером. У настоящего Deep SORT признак — обученный 128-мерный дескриптор человека, а не цвет. [Статья Deep SORT, §2](https://arxiv.org/pdf/1703.07402), [авторский порядок сопоставления](https://github.com/nwojke/deep_sort/blob/master/deep_sort/tracker.py#L84-L118), [галерея](https://github.com/nwojke/deep_sort/blob/master/deep_sort/nn_matching.py#L72-L142).

In [ ]:
# Код стенда — независимая учебная реализация порядка решений Deep SORT.
# Требует make_crossing_scene, box_iou_matrix, rgb_feature из предыдущего стенда.

def color_xyah(box):
    x1,y1,x2,y2=np.asarray(box,float)
    w=max(float(x2-x1),1e-6);h=max(float(y2-y1),1e-6)
    return np.array([(x1+x2)/2,(y1+y2)/2,w/h,h],float)

def color_box(x):
    u,v,a,h=np.asarray(x[:4],float);h=max(float(h),1e-3);w=max(float(a*h),1e-3)
    return np.array([u-w/2,v-h/2,u+w/2,v+h/2],float)

def color_init(box,feature,track_id,n_init):
    z=color_xyah(box);h=z[3]
    std=np.array([2*h/20,2*h/20,.1,2*h/20,10*h/160,10*h/160,.001,10*h/160])
    x=np.r_[z,np.zeros(4)]
    tr={'id':int(track_id),'x':x,'P':np.diag(std**2),'hits':1,'age':1,
        'time_since_update':0,'status':'confirmed' if n_init<=1 else 'tentative',
        'pending':[feature.copy()],'gallery':[]}
    if n_init<=1:tr['gallery']=[feature.copy()];tr['pending']=[]
    return tr

def color_predict(tr):
    F=np.eye(8);F[:4,4:]=np.eye(4)
    h=max(float(tr['x'][3]),1.)
    std=np.array([h/20,h/20,.01,h/20,h/160,h/160,.0001,h/160])
    tr['x']=F@tr['x'];tr['P']=F@tr['P']@F.T+np.diag(std**2)
    tr['P']=(tr['P']+tr['P'].T)/2
    tr['age']+=1;tr['time_since_update']+=1

def color_project(tr):
    H=np.eye(4,8);h=max(float(tr['x'][3]),1.)
    std=np.array([h/20,h/20,.1,h/20])
    return tr['x'][:4],tr['P'][:4,:4]+np.diag(std**2)

def color_mahalanobis2(tr,box):
    mu,S=color_project(tr);delta=color_xyah(box)-mu
    return float(delta@np.linalg.solve(S,delta))

def color_update(tr,box,feature,n_init,budget):
    z=color_xyah(box);mu,S=color_project(tr);H=np.eye(4,8)
    K=np.linalg.solve(S,(tr['P']@H.T).T).T
    residual=z-mu;tr['x']=tr['x']+K@residual
    I=np.eye(8);A=I-K@H;R=S-H@tr['P']@H.T
    tr['P']=A@tr['P']@A.T+K@R@K.T;tr['P']=(tr['P']+tr['P'].T)/2
    tr['hits']+=1;tr['time_since_update']=0
    if tr['status']=='tentative':
        tr['pending'].append(feature.copy())
        if tr['hits']>=n_init:
            tr['status']='confirmed'
            tr['gallery']=(tr['gallery']+tr['pending'])[-budget:]
            tr['pending']=[]
    else:tr['gallery']=(tr['gallery']+[feature.copy()])[-budget:]

def color_assign(cost,valid,track_rows,det_cols,unmatched_cost):
    # Явные фиктивные столбцы показывают право оставить track без пары.
    if not track_rows or not det_cols:return [],list(track_rows),list(det_cols)
    m=len(track_rows);n=len(det_cols)
    sub=np.array(cost[np.ix_(track_rows,det_cols)],float)
    ok=np.asarray(valid[np.ix_(track_rows,det_cols)],bool)
    work=np.full((m,n+m),float(unmatched_cost)+1e-4)
    work[:,:n]=np.where(ok,sub,1e6)
    ri,cj=linear_sum_assignment(work)
    pairs=[(track_rows[int(i)],det_cols[int(j)]) for i,j in zip(ri,cj)
           if j<n and ok[i,j] and work[i,j]<=unmatched_cost+1e-5]
    used_r={i for i,_ in pairs};used_c={j for _,j in pairs}
    return pairs,[i for i in track_rows if i not in used_r],[j for j in det_cols if j not in used_c]

class ColorGalleryTracker:
    def __init__(self,max_age=3,n_init=3,budget=4,color_threshold=.35,
                 gate_threshold=9.4877,use_gate=True,use_cascade=True,
                 use_iou_fallback=True,min_iou=.3):
        self.tracks=[];self.next_id=1
        self.max_age=int(max_age);self.n_init=int(n_init);self.budget=int(budget)
        self.color_threshold=float(color_threshold);self.gate_threshold=float(gate_threshold)
        self.use_gate=bool(use_gate);self.use_cascade=bool(use_cascade)
        self.use_iou_fallback=bool(use_iou_fallback);self.min_iou=float(min_iou)

    def step(self,detections):
        detections=list(detections);n=len(detections)
        features=[np.asarray(rgb_feature(d['patch']),float) for d in detections]
        features=[f/max(np.linalg.norm(f),1e-12) for f in features]
        for tr in self.tracks:color_predict(tr)
        predicted=[{'id':tr['id'],'box':color_box(tr['x']).copy(),
                    'status':tr['status'],'time_since_update':tr['time_since_update']}
                   for tr in self.tracks]
        nt=len(self.tracks)
        maha=np.array([[color_mahalanobis2(tr,d['box']) for d in detections]
                       for tr in self.tracks],float).reshape(nt,n)
        motion_ok=(maha<=self.gate_threshold) if self.use_gate else np.ones((nt,n),bool)
        app=np.full((nt,n),np.nan,float)
        for i,tr in enumerate(self.tracks):
            if tr['status']=='confirmed' and tr['gallery']:
                app[i,:]=[min(float(1-np.clip(g@f,-1,1)) for g in tr['gallery'])
                          for f in features]
        app_valid=(app<=self.color_threshold)&motion_ok
        all_det=list(range(n));remaining=all_det[:];pairs=[];stages=[]
        confirmed=[i for i,tr in enumerate(self.tracks)
                   if tr['status']=='confirmed' and tr['time_since_update']<=self.max_age]
        ages=sorted({self.tracks[i]['time_since_update'] for i in confirmed})
        groups=[[i for i in confirmed if self.tracks[i]['time_since_update']==age]
                for age in ages] if self.use_cascade else [confirmed]
        for level,rows in enumerate(groups):
            if not rows or not remaining:continue
            matched,_,remaining=color_assign(app,app_valid,rows,remaining,self.color_threshold)
            for i,j in matched:
                pairs.append((i,j,'appearance'))
                stages.append(f'appearance age={self.tracks[i]["time_since_update"]}'
                              if self.use_cascade else 'appearance global')
        used_tracks={i for i,_,_ in pairs}
        tentative=[i for i,tr in enumerate(self.tracks) if tr['status']=='tentative']
        recent=[i for i in confirmed if i not in used_tracks and
                self.tracks[i]['time_since_update']==1]
        iou_candidates=(tentative+recent) if self.use_iou_fallback else []
        iou=np.zeros((nt,n),float)
        if nt and n:
            iou=box_iou_matrix(np.array([color_box(tr['x']) for tr in self.tracks]),
                               np.array([d['box'] for d in detections]))
        if iou_candidates and remaining:
            iou_pairs,_,remaining=color_assign(1-iou,iou>=self.min_iou,
                                               iou_candidates,remaining,1-self.min_iou)
            for i,j in iou_pairs:
                pairs.append((i,j,'IoU'))
                stages.append('IoU tentative' if i in tentative else 'IoU recent fallback')
        det_to_id=[None]*n;matched_rows={i for i,_,_ in pairs}
        match_info=[]
        for (i,j,stage),label in zip(pairs,stages):
            tr=self.tracks[i];tid=tr['id']
            color_update(tr,detections[j]['box'],features[j],self.n_init,self.budget)
            det_to_id[j]=tid
            match_info.append({'track_id':tid,'detection':j,'stage':label,
                               'color_cost':None if np.isnan(app[i,j]) else float(app[i,j]),
                               'mahalanobis2':float(maha[i,j]),'iou':float(iou[i,j])})
        deleted=[]
        for i,tr in enumerate(self.tracks):
            if i in matched_rows:continue
            if tr['status']=='tentative' or tr['time_since_update']>self.max_age:
                tr['status']='deleted';deleted.append(tr['id'])
        self.tracks=[tr for tr in self.tracks if tr['status']!='deleted']
        created=[]
        for j in remaining:
            tr=color_init(detections[j]['box'],features[j],self.next_id,self.n_init)
            self.tracks.append(tr);det_to_id[j]=tr['id'];created.append(tr['id'])
            self.next_id+=1
        internal=[{'id':tr['id'],'box':color_box(tr['x']).copy(),
                   'status':tr['status'],'hits':tr['hits'],'age':tr['age'],
                   'time_since_update':tr['time_since_update'],
                   'gallery':[g.copy() for g in tr['gallery']]}
                  for tr in self.tracks]
        return {'predicted':predicted,'appearance_cost':app,'mahalanobis2':maha,
                'gate_allowed':motion_ok,'iou':iou,'matches':match_info,
                'unmatched_detections':remaining,'created':created,'deleted':deleted,
                'det_to_id':det_to_id,'internal':internal,
                'pairs':[(m['track_id'],m['detection'],m['stage']) for m in match_info]}

def color_light_shift(frames,amount):
    # GT задаёт только генерацию известного сбоя признака и никогда не входит в tracker.step.
    if amount<=0:return frames
    for t,frame in enumerate(frames):
        if 3<=t<=6:
            for det in frame['detections']:
                if det['gt']==0:
                    det['patch']=np.clip((1-amount)*det['patch']+
                                         amount*np.array([.45,.45,.45]),0,1)
    return frames

def run_color_scene(frames,**kwargs):
    tracker=ColorGalleryTracker(**kwargs);steps=[];previous={};changes=0
    for t,frame in enumerate(frames):
        diag=tracker.step(frame['detections']);diag['frame']=t
        new_changes=[]
        for j,det in enumerate(frame['detections']):
            gt=int(det['gt']);tid=diag['det_to_id'][j]
            if gt<0 or tid is None:continue
            if gt in previous and previous[gt]!=tid:
                changes+=1;new_changes.append((gt,previous[gt],tid))
            previous[gt]=tid
        diag['identity_changes']=changes;diag['new_changes']=new_changes
        steps.append(diag)
    return steps

def color_sort_id_changes(frames,sort_steps):
    previous={};changes=0;out=[]
    for frame,diag in zip(frames,sort_steps):
        ids=diag.get('det_to_id',{})
        for j,det in enumerate(frame['detections']):
            gt=int(det['gt']);tid=ids.get(j) if isinstance(ids,dict) else (ids[j] if j<len(ids) else None)
            if gt<0 or tid is None:continue
            if gt in previous and previous[gt]!=tid:changes+=1
            previous[gt]=tid
        out.append(changes)
    return out

def color_show(frame=8,color_mode='distinct',miss_length=1,jitter=0.,
               false_positive=False,max_age=3,n_init=3,budget=4,
               color_threshold=.35,gate_threshold=9.5,use_gate=True,
               use_cascade=True,use_iou_fallback=True,light_shift=0.):
    frames=make_crossing_scene(miss_length=miss_length,jitter=jitter,
                               color_mode=color_mode,false_positive=false_positive)
    frames=color_light_shift(frames,light_shift)
    params=dict(max_age=max_age,n_init=n_init,budget=budget,
                color_threshold=color_threshold,gate_threshold=gate_threshold,
                use_gate=use_gate,use_cascade=use_cascade,
                use_iou_fallback=use_iou_fallback)
    steps=run_color_scene(frames,**params)
    sort_steps=run_sort_scene(frames,iou_threshold=.3,max_age=max_age,
                              min_hits=n_init,use_prediction=True)
    sort_changes=color_sort_id_changes(frames,sort_steps)
    t=int(np.clip(frame,0,len(frames)-1));scene=frames[t];diag=steps[t];sdiag=sort_steps[t]
    fig=plt.figure(figsize=(11.6,7.5),constrained_layout=True)
    grid=fig.add_gridspec(2,2,height_ratios=[1.2,1],width_ratios=[1.35,1])
    ax=fig.add_subplot(grid[0,0]);time_ax=fig.add_subplot(grid[0,1])
    mat_ax=fig.add_subplot(grid[1,0]);info_ax=fig.add_subplot(grid[1,1])
    palette=['#2255AA','#C23B22','#238B45','#7D3C98','#A67112','#167D88']
    for gt,box in scene['truth'].items():
        box=np.asarray(box,float);ax.add_patch(Rectangle(box[:2],box[2]-box[0],box[3]-box[1],
                         fill=False,edgecolor='0.65',lw=1.5,ls=':'))
        ax.text((box[0]+box[2])/2,box[3]+1.2,f'GT {"AB"[gt]}',ha='center',color='0.45',fontsize=8)
    for item in diag['predicted']:
        box=item['box'];ax.add_patch(Rectangle(box[:2],box[2]-box[0],box[3]-box[1],
                                      fill=False,edgecolor=palette[item['id']%len(palette)],
                                      lw=1.4,ls='--',alpha=.6))
    for j,det in enumerate(scene['detections']):
        box=np.asarray(det['box'],float);rgb=np.clip(np.asarray(det['patch']).mean((0,1)),0,1)
        tid=diag['det_to_id'][j];col=palette[tid%len(palette)] if tid is not None else 'black'
        ax.add_patch(Rectangle(box[:2],box[2]-box[0],box[3]-box[1],
                              facecolor=rgb,edgecolor=col,alpha=.55,lw=2))
        ax.text(box[0]+1,box[1]-2,f'D{j} → ID {tid}',color=col,fontsize=8,weight='bold')
    ax.set(xlim=(0,100),ylim=(25,80),xlabel='x, pixels',ylabel='y, pixels',
           title=f'Кадр {t}: заполнение — RGB; рамка — назначенный ID')
    ax.grid(alpha=.15)
    for gt,y in [(0,1),(1,0)]:
        for series,off,marker,name in [(steps,.12,'o','Color'),(sort_steps,-.12,'x','SORT')]:
            xs=[];labels=[]
            for k,d in enumerate(series[:t+1]):
                for j,det in enumerate(frames[k]['detections']):
                    if det['gt']==gt:
                        ids=d['det_to_id']
                        tid=ids.get(j) if isinstance(ids,dict) else (ids[j] if j<len(ids) else None)
                        if tid is not None:xs.append(k);labels.append(tid)
            time_ax.scatter(xs,[y+off]*len(xs),marker=marker,
                            color='#176BA0' if name=='Color' else '#D75A30',s=50,
                            label=name if gt==0 else None)
            for xval,tid in zip(xs,labels):
                time_ax.text(xval,y+off+.07,str(tid),ha='center',fontsize=7)
    time_ax.set(xlim=(-.5,len(frames)-.5),ylim=(-.4,1.5),xticks=range(len(frames)),
                yticks=[0,1],yticklabels=['GT B','GT A'],xlabel='frame',
                title='ID по кадрам (вкл. tentative)')
    time_ax.legend(loc='upper right',fontsize=8);time_ax.grid(axis='x',alpha=.2)
    A=diag['appearance_cost'];G=diag['gate_allowed'];D=diag['mahalanobis2']
    if A.size and A.shape[1] and np.isfinite(A).any():
        shown=np.where(np.isfinite(A),A,1.)
        mat_ax.imshow(shown,cmap='YlGnBu',vmin=0,vmax=1)
        for i,j in np.ndindex(A.shape):
            value='—' if not np.isfinite(A[i,j]) else f'{A[i,j]:.2f}'
            mark='' if G[i,j] else ' ×'
            mat_ax.text(j,i,value+mark,ha='center',va='center',fontsize=9,
                        color='crimson' if mark else 'black')
        for match in diag['matches']:
            if 'appearance' not in match['stage']:continue
            row=next((i for i,p in enumerate(diag['predicted']) if p['id']==match['track_id']),None)
            if row is not None:mat_ax.add_patch(Rectangle((match['detection']-.46,row-.46),.92,.92,
                                             fill=False,lw=2.5,ec='limegreen'))
        mat_ax.set(xticks=range(A.shape[1]),xticklabels=[f'D{j}' for j in range(A.shape[1])],
                   yticks=range(A.shape[0]),yticklabels=[f'ID {p["id"]} ({p["time_since_update"]})' for p in diag['predicted']],
                   title='min cosine; × = запрет Mahalanobis gate')
    else:
        mat_ax.text(.5,.5,'Нет подтверждённых треков или детекций\nдля цветового назначения',
                    ha='center',va='center',transform=mat_ax.transAxes)
        mat_ax.set_axis_off()
    info_ax.set_axis_off()
    lines=['Этапы назначения:']
    for m in diag['matches'][:3]:
        c='—' if m['color_cost'] is None else f'{m["color_cost"]:.2f}'
        lines.append(f'ID {m["track_id"]}–D{m["detection"]}: {m["stage"]}, c={c}, d²={m["mahalanobis2"]:.2f}')
    if not diag['matches']:lines.append('нет пар')
    lines.append(f'Созданы {diag["created"]}; удалены {diag["deleted"]}')
    info_ax.text(0,.98,'\n'.join(lines),va='top',fontsize=9,family='monospace',
                 transform=info_ax.transAxes)
    info_ax.text(0,.51,'Галерея: от старых к новым',va='center',fontsize=9,
                 transform=info_ax.transAxes)
    for row,tr in enumerate(diag['internal'][:3]):
        y=.40-.15*row;gallery=tr['gallery']
        info_ax.text(0,y,f'ID {tr["id"]} {tr["status"]} miss={tr["time_since_update"]} G={len(gallery)}',
                     va='center',fontsize=9,transform=info_ax.transAxes)
        for k,g in enumerate(gallery[:6]):
            rgb=np.clip(g/max(float(np.max(g)),1e-12),0,1)
            info_ax.add_patch(Rectangle((.58+.065*k,y-.025),.05,.05,
                              facecolor=rgb,edgecolor='0.25',lw=.5,
                              transform=info_ax.transAxes))
    plt.show();plt.close(fig)
    print('Текущий кадр: пары',diag['pairs'],'| Color ID',diag['det_to_id'],
          '| SORT ID',sdiag['det_to_id'],'| смены',diag['identity_changes'],sort_changes[t])
    print('Mahalanobis²:\n',np.array2string(D,precision=2),
          '\nGate:\n',G.astype(int),'\nColor cost:\n',np.array2string(A,precision=2))
    print('Внутреннее состояние и галерея (нормированный RGB):')
    for tr in diag['internal']:
        gallery=[np.round(g,2).tolist() for g in tr['gallery']]
        print(f'ID {tr["id"]}: {tr["status"]}, miss={tr["time_since_update"]}, '
              f'hits={tr["hits"]}, G={gallery}')
    return {'frame':t,'pairs':diag['pairs'],'det_to_id':diag['det_to_id'],
            'sort_det_to_id':sdiag['det_to_id'],'identity_changes':diag['identity_changes'],
            'sort_identity_changes':sort_changes[t],'internal':diag['internal'],
            'appearance_cost':A,'mahalanobis2':D,'gate_allowed':G,
            'matches':diag['matches'],'created':diag['created'],'deleted':diag['deleted'],
            'frames':frames,'color_steps':steps,'sort_steps':sort_steps}

color_defaults={'frame':8,'color_mode':'distinct','miss_length':1,'jitter':0.,
                'false_positive':False,'max_age':3,'n_init':3,'budget':4,
                'color_threshold':.35,'gate_threshold':9.5,'use_gate':True,
                'use_cascade':True,'use_iou_fallback':True,'light_shift':0.}
color_controls={
    'frame':W.IntSlider(value=8,min=0,max=11,step=1,description='Кадр',continuous_update=True),
    'color_mode':W.Dropdown(options=[('Различимые','distinct'),('Одинаковые','same'),
                                     ('Цвета поменялись','swapped'),('Серые','gray')],
                            value='distinct',description='Цвет'),
    'miss_length':W.IntSlider(value=1,min=0,max=3,step=1,description='Пропуск',continuous_update=True),
    'jitter':W.FloatSlider(value=0,min=0,max=4,step=.5,description='Шум',continuous_update=True),
    'false_positive':W.Checkbox(value=False,description='Ложная детекция'),
    'max_age':W.IntSlider(value=3,min=1,max=5,step=1,description='max_age',continuous_update=True),
    'n_init':W.IntSlider(value=3,min=1,max=4,step=1,description='n_init',continuous_update=True),
    'budget':W.IntSlider(value=4,min=1,max=6,step=1,description='Галерея',continuous_update=True),
    'color_threshold':W.FloatSlider(value=.35,min=.05,max=.9,step=.05,description='Порог цвета',continuous_update=True),
    'gate_threshold':W.FloatSlider(value=9.5,min=2,max=30,step=.5,description='Порог d²',continuous_update=True),
    'use_gate':W.Checkbox(value=True,description='Mahalanobis gate'),
    'use_cascade':W.Checkbox(value=True,description='Каскад'),
    'use_iou_fallback':W.Checkbox(value=True,description='IoU fallback'),
    'light_shift':W.FloatSlider(value=0,min=0,max=.9,step=.1,description='Обесцвечивание',continuous_update=True),
}
color_play=W.Play(value=8,min=0,max=11,step=1,interval=750,description='▶')
color_link=W.link((color_play,'value'),(color_controls['frame'],'value'))
color_prev=W.Button(description='← Кадр');color_next=W.Button(description='Кадр →')
color_reset=W.Button(description='Сбросить',icon='refresh')
color_output=W.Output();color_state={'calls':0,'last':None}
def color_refresh(change=None):
    with capture_output(display=True) as captured:
        result=color_show(**{key:widget.value for key,widget in color_controls.items()})
    items=[{'output_type':'display_data','data':item.data,'metadata':item.metadata}
           for item in captured.outputs]
    if captured.stdout:
        items.append({'output_type':'stream','name':'stdout','text':captured.stdout})
    if captured.stderr:
        items.append({'output_type':'stream','name':'stderr','text':captured.stderr})
    color_output.outputs=tuple(items)
    color_state['last']=result;color_state['calls']+=1
for widget in color_controls.values():widget.observe(color_refresh,names='value')
def color_prev_click(button):color_controls['frame'].value=max(0,color_controls['frame'].value-1)
def color_next_click(button):color_controls['frame'].value=min(11,color_controls['frame'].value+1)
def color_reset_click(button):
    changed=False
    for key,default in color_defaults.items():
        if color_controls[key].value!=default:
            color_controls[key].value=default;changed=True
    if not changed:color_refresh()
color_prev.on_click(color_prev_click);color_next.on_click(color_next_click)
color_reset.on_click(color_reset_click)
color_buttons={'prev':color_prev,'next':color_next,'reset':color_reset,'play':color_play}
MOTION_PANELS['color']={'controls':color_controls,'state':color_state,
                       'buttons':color_buttons,'refresh':color_refresh,
                       'output':color_output,'play_link':color_link}
display(W.VBox([
    W.HBox([color_play,color_prev,color_controls['frame'],color_next,color_reset]),
    W.HBox([color_controls[k] for k in ('color_mode','miss_length','jitter','false_positive')],
           layout=W.Layout(flex_flow='row wrap')),
    W.HBox([color_controls[k] for k in ('max_age','n_init','budget','light_shift')],
           layout=W.Layout(flex_flow='row wrap')),
    W.HBox([color_controls[k] for k in ('color_threshold','gate_threshold','use_gate',
                                        'use_cascade','use_iou_fallback')],
           layout=W.Layout(flex_flow='row wrap')),
    color_output]))
color_refresh()

### Проверка на стенде 8

1. **Движение и идентичность.** При одинаковых параметрах сравните «различимые» и «одинаковые» цвета на кадре после пересечения. До переключения предскажите, какие пары останутся допустимыми по движению, затем запишите ID обоих объектов и число смен ID у двух трекеров. Объясните, почему цвет не восстанавливает личность при одинаковых признаках.
2. **Пропуск и срок жизни.** Поставьте длину пропуска 1, затем 3, `max_age` сначала 1, затем 4. На каком кадре пропавший объект получает прежний ID, а когда создаётся новый? Укажите `time_since_update` перед повторным обнаружением и состояние после него. Почему даже идеальный цвет не возвращает удалённый ID?
3. **Проверка движения и каскад.** При «цвета поменялись», пропуске 1 и кадре 7 сравните разрешённые пары с включённым и отключённым gate: для ID 1 цветовая стоимость равна нулю, но $d^2$ превышает порог. Затем отдельно поставьте «одинаковые» цвета, пропуск 2, `n_init=2`, `max_age=4`, порог цвета $0.5$, кадр 6. Сравните назначения с каскадом и без него: какой трек получает единственную детекцию и как меняется число смен ID к кадру 7?
4. **История внешности и резервный IoU.** Установите «различимые» цвета, обесцвечивание $0.8$, порог цвета $0.1$, кадр 7; сравните размер галереи 1 и 6, не меняя остальные настройки. Выпишите стоимость ID 1 → детекция объекта A, этап решения и число смен ID. По образцам галереи объясните, почему старый цвет помогает после возврата исходного цвета. Затем отключите резервный IoU: какие tentative теряют шанс подтвердиться? Назовите кадр и стадию решения.

Все выводы делайте по числам и ID на стенде. Число смен **назначенных** ID учитывает даже tentative; это диагностическая величина этой синтетической сцены, не метрика MOTChallenge.

## Дальше
В 02 реализуйте общие механизмы и свяжите их в трекер, проведите контролируемый опыт и отдельно посмотрите реальное видео. Синтетика дает точную GT и изолирует association; реальный foreground показывает границы измерений.

Источники: [SORT](https://arxiv.org/abs/1602.00763), [DeepSORT](https://arxiv.org/abs/1703.07402), [авторский Kalman DeepSORT](https://github.com/nwojke/deep_sort/blob/master/deep_sort/kalman_filter.py).